# cuda-empty-cache — worked example 1: Release cache once per N rendered frames

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cuda-empty-cache`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a long render loop the CUDA caching allocator holds onto freed blocks so it can reuse them quickly. That reserved memory can build up and trigger an OOM even though live tensors are small. Calling `torch.cuda.empty_cache()` at a fixed cadence returns those cached-but-unused blocks to the driver. The call never destroys a live tensor — only blocks with no references are released.

## Worked solution

**Goal.** Produce one scalar 'brightness' per frame (the mean of the frame tensor) while calling `empty_cache()` every `K` frames.

**Step 1 — iterate with a 1-indexed counter.** We use `enumerate(frames, start=1)` so frame 1 is the first frame. This matters because 'every K frames' is naturally 1-indexed: with `K=4` we want to release after frames 4, 8, 12.

**Step 2 — do the per-frame work.** For each frame we compute `f.mean()`, which is the scalar work the loop exists to do. We append it to `results`.

**Step 3 — release on the cadence.** `if i % K == 0:` is true exactly on the multiples of `K`, so the release fires at the right cadence. The release does NOT touch `results` — those scalars are live tensors with references, so the allocator leaves them alone. That is the key safety property of `empty_cache()`.

**Step 4 — stack into a 1-D tensor.** `t.stack(results)` turns the list of 0-D scalars into a length-`len(frames)` 1-D tensor, the canonical return shape.

Because we run on CPU here the call is a harmless no-op, but the logic is identical on GPU.

In [ ]:
def render_loop_release(frames, K: int):
    results = []
    for i, f in enumerate(frames, start=1):
        results.append(f.mean())
        if i % K == 0:
            t.cuda.empty_cache()
    return t.stack(results)

t.manual_seed(0)
frames = [t.randn(3, 8, 8) for _ in range(6)]
out = render_loop_release(frames, K=2)
print(out.shape)
print(t.allclose(out, t.stack([f.mean() for f in frames])))